In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
import time

In [1]:

# --- CONFIGURATION ---
TOKEN = "ee92dbba-157d-4edb-a4fd-02a5e23f4279"
HEADERS = {"Authorization": f"Bearer {TOKEN}"} if TOKEN else {}
BASE_URL = "https://boardgamegeek.com/xmlapi2/thing"
# Pro-tip: Start with a small range to test (e.g., 1 to 100)
# Top games like Gloomhaven are in the 174430+ range.
GAME_IDS = range(1, 1001) 
CHUNK_SIZE = 20
OUTPUT_FILE = "bgg_games_data.csv"

def fetch_bgg_data(ids):
    id_string = ",".join(map(str, ids))
    params = {"id": id_string, "stats": 1}
    
    while True:
        response = requests.get(BASE_URL, params=params, headers=HEADERS)
        if response.status_code == 200:
            return response.content
        elif response.status_code == 202:
            print("  BGG is preparing data... waiting 5 seconds.")
            time.sleep(5)
        else:
            print(f"  Error {response.status_code}. Skipping chunk.")
            return None

def parse_xml(xml_data):
    games_list = []
    root = ET.fromstring(xml_data)
    
    for item in root.findall("item"):
        if item.get("type") != "boardgame":
            continue
            
        game_id = item.get("id")
        name = item.find("name[@type='primary']").get("value")
        
        # Statistics
        stats = item.find("statistics/ratings")
        complexity = stats.find("averageweight").get("value")
        owned = stats.find("owned").get("value")
        if int(owned) < 1000: 
            continue
        
        avg_rating = stats.find("average").text if stats.find("average") is not None else "0"
        bayes_avg = stats.find("bayesaverage").text if stats.find("bayesaverage") is not None else "0"
        users_rated = stats.find("usersrated").text if stats.find("usersrated") is not None else "0"
        
        
        # Mechanics (Multiple links)
        mechanics = [link.get("value") for link in item.findall("link[@type='boardgamemechanic']")]
        
        games_list.append({
            "game_id": game_id,
            "name": name,
            "complexity": complexity,
            "owned": owned,
            "avg_rating": avg_rating,
            "bayes_avg": bayes_avg,
            "users_rated": users_rated,
            "mechanics": "|".join(mechanics)
        })

    return games_list

# --- MAIN EXECUTION ---
all_data = []

for i in range(0, len(GAME_IDS), CHUNK_SIZE):
    chunk = list(GAME_IDS)[i:i + CHUNK_SIZE]
    print(f"Fetching IDs {chunk[0]} to {chunk[-1]}...")
    
    raw_xml = fetch_bgg_data(chunk)
    if raw_xml:
        parsed_chunk = parse_xml(raw_xml)
        all_data.extend(parsed_chunk)
    
    # Respect the API: pause between chunks
    time.sleep(1)

# Save to CSV
df = pd.DataFrame(all_data)
df.to_csv(OUTPUT_FILE, index=False)
print(f"Finished! Data saved to {OUTPUT_FILE}")

Fetching IDs 1 to 20...
Fetching IDs 21 to 40...
Fetching IDs 41 to 60...
Fetching IDs 61 to 80...
Fetching IDs 81 to 100...
Fetching IDs 101 to 120...
Fetching IDs 121 to 140...
Fetching IDs 141 to 160...
Fetching IDs 161 to 180...
Fetching IDs 181 to 200...
Fetching IDs 201 to 220...
Fetching IDs 221 to 240...
Fetching IDs 241 to 260...
Fetching IDs 261 to 280...
Fetching IDs 281 to 300...
Fetching IDs 301 to 320...
Fetching IDs 321 to 340...
Fetching IDs 341 to 360...
Fetching IDs 361 to 380...
Fetching IDs 381 to 400...
Fetching IDs 401 to 420...
Fetching IDs 421 to 440...
Fetching IDs 441 to 460...
Fetching IDs 461 to 480...
Fetching IDs 481 to 500...
Fetching IDs 501 to 520...
Fetching IDs 521 to 540...
Fetching IDs 541 to 560...
Fetching IDs 561 to 580...
Fetching IDs 581 to 600...
Fetching IDs 601 to 620...
Fetching IDs 621 to 640...
Fetching IDs 641 to 660...
Fetching IDs 661 to 680...
Fetching IDs 681 to 700...
Fetching IDs 701 to 720...
Fetching IDs 721 to 740...
Fetching ID